# 📊 Job Trend Tracker — Documentación del Proyecto
**Proyecto de Ingeniería de Datos con Python y MCP**

Este notebook explica la estructura completa del proyecto, qué hace cada carpeta y cada archivo, y cómo se conectan entre sí para formar el pipeline de datos.

---

## 🎯 ¿Qué hace este proyecto?

**Job Trend Tracker** es un pipeline de datos que:

1. **Extrae** ofertas de trabajo de sitios como LinkedIn e Indeed mediante scraping
2. **Transforma** esos datos: los limpia y extrae las habilidades (skills) mencionadas
3. **Almacena** todo en una base de datos DuckDB local
4. **Analiza** qué skills están creciendo o bajando semana a semana
5. **Expone** esa información a través de un servidor MCP para que un agente de IA (Claude) pueda responder preguntas en lenguaje natural
6. **Visualiza** las tendencias en un dashboard interactivo con Streamlit

### Pregunta que responde el proyecto:
> *¿Qué habilidades tecnológicas están siendo más demandadas en el mercado laboral ahora mismo?*

---
## 🗂️ Estructura General del Proyecto

```
job_trend_tracker/
│
├── data/
│   ├── raw/
│   ├── processed/
│   └── external/
│
├── notebooks/
│
├── src/
│   ├── ingestion/
│   ├── transformation/
│   ├── pipeline/
│   └── utils/
│
├── mcp_server/
├── dashboard/
├── config/
├── tests/
│
├── requirements.txt
├── .env
├── .gitignore
└── README.md
```

---
## 📁 data/

Esta es la carpeta más importante del proyecto. Aquí viven **todos los datos** en sus diferentes estados de procesamiento. Se divide en tres subcarpetas que representan el ciclo de vida del dato.

### 📁 data/raw/
Contiene los datos **tal y como fueron extraídos**, sin ninguna modificación. 

- Son archivos CSV o JSON que vienen directamente del scraper
- **Nunca se modifican a mano** — si algo sale mal, se vuelve a correr el scraper
- Sirven como respaldo: si la transformación falla, siempre tienes los datos originales
- Ejemplo de archivo: `jobs_linkedin_2024_01_15.csv`

### 📁 data/processed/
Contiene los datos ya **limpios y transformados**, listos para ser analizados.

- Los datos están estandarizados (fechas en el mismo formato, texto sin caracteres raros)
- Las skills ya fueron extraídas de las descripciones de trabajo
- Es el input del dashboard y del servidor MCP
- Ejemplo de archivo: `jobs_clean.parquet`, `skills_by_week.csv`

### 📁 data/external/
Contiene datos de referencia que **no generamos nosotros**, sino que se importan.

- `skills_list.csv` — lista maestra de habilidades tech que buscamos en las ofertas (Python, SQL, Airflow, dbt, etc.)
- Podría contener también clasificaciones de roles, lista de ciudades, etc.

---
## 📁 notebooks/

Contiene los **Jupyter Notebooks** usados para exploración y análisis. Son el espacio de trabajo experimental del proyecto.

- `01_exploracion.ipynb` — Primer vistazo a los datos crudos: ¿cuántas filas hay?, ¿qué columnas?, ¿hay valores nulos?
- `02_analisis_tendencias.ipynb` — Análisis visual de tendencias: gráficas de skills más mencionadas, evolución semanal
- `00_documentacion_proyecto.ipynb` — Este notebook: documentación completa del proyecto

> Los notebooks son para **explorar y entender** los datos. El código definitivo siempre va en la carpeta `src/`.

---
## 📁 src/

El corazón del proyecto. Contiene **todo el código fuente** organizado por responsabilidad. Cada subcarpeta tiene una función específica y bien definida.

### 📁 src/ingestion/
Responsable de **traer los datos desde el exterior** al proyecto.

- `scraper_linkedin.py` — Extrae ofertas de trabajo de LinkedIn usando requests/Playwright. Guarda el resultado en `data/raw/`
- `scraper_indeed.py` — Hace lo mismo pero con Indeed
- `__init__.py` — Archivo que le dice a Python que esta carpeta es un módulo importable

Analogía: es como el camión de reparto que trae la materia prima a la fábrica.

---

### 📁 src/transformation/
Responsable de **limpiar y transformar** los datos crudos.

- `clean_jobs.py` — Limpia el dataset: elimina duplicados, estandariza fechas, elimina caracteres especiales, maneja valores nulos
- `extract_skills.py` — Lee cada descripción de trabajo y detecta qué skills menciona (ej: si dice "experience with Python" → extrae "Python")

Analogía: es la línea de producción que procesa la materia prima.

---

### 📁 src/pipeline/
Responsable de **orquestar el flujo completo** de extremo a extremo.

- `run_pipeline.py` — Script principal que ejecuta los pasos en orden: primero ingestion, luego transformation, luego guarda en DuckDB. Se puede programar para correr diariamente.

Analogía: es el director de la fábrica que coordina todos los pasos en orden.

---

### 📁 src/utils/
Funciones auxiliares **reutilizables** en todo el proyecto.

- `db.py` — Funciones para conectarse a DuckDB: crear tablas, insertar datos, hacer queries
- `helpers.py` — Funciones pequeñas de apoyo: formatear fechas, limpiar texto, logging

---
## 📁 mcp_server/

Esta carpeta es la que hace el proyecto **único e innovador**. Contiene el servidor MCP (Model Context Protocol) que le permite a un agente de IA como Claude interactuar directamente con los datos.

- `server.py` — Define y levanta el servidor MCP. Es el punto de entrada cuando Claude se conecta al proyecto
- `tools.py` — Define las **herramientas (tools)** que el agente puede usar. Cada tool es una función que Claude puede llamar. Ejemplos:
  - `get_top_skills(n, week)` → devuelve las N skills más demandadas de una semana
  - `get_skill_trend(skill_name)` → devuelve la evolución histórica de una skill
  - `compare_skills(skill_a, skill_b)` → compara la demanda de dos skills

### ¿Cómo funciona MCP?
MCP es un protocolo que permite a Claude conectarse a herramientas y fuentes de datos externas. En lugar de que Claude tenga que adivinar las tendencias del mercado, puede consultar directamente nuestra base de datos con datos reales y actualizados.

```
Usuario → Claude → MCP Server → DuckDB → Respuesta con datos reales
```

---
## 📁 dashboard/

Contiene la **aplicación visual** del proyecto construida con Streamlit.

- `app.py` — Archivo principal de la app. Define las páginas, filtros y layout general del dashboard
- `charts.py` — Funciones que generan las gráficas: barras de top skills, líneas de tendencia semanal, heatmaps de combinaciones de skills

El dashboard permite explorar visualmente qué skills están subiendo o bajando en el mercado laboral, filtrando por fecha, país o tipo de rol.

---
## 📁 config/

Centraliza toda la **configuración del proyecto** para que sea fácil de cambiar sin tocar el código.

- `settings.yaml` — Configuración general: cuántas ofertas extraer, cada cuánto correr el pipeline, qué keywords buscar, nombre de la base de datos
- `.env.example` — Plantilla de las variables de entorno necesarias (sin los valores reales). Sirve de guía para que otros developers sepan qué variables configurar

> Las credenciales reales (API keys, contraseñas) nunca van en el código, siempre en el archivo `.env` que está en el `.gitignore`

---
## 📁 tests/

Contiene las **pruebas automáticas** del código para asegurar que todo funciona correctamente.

- `test_scraper.py` — Verifica que el scraper retorna datos con el formato correcto, que maneja errores de red, etc.
- `test_transform.py` — Verifica que la limpieza y extracción de skills funciona bien con diferentes casos de entrada

Los tests son esenciales para saber que el pipeline no está roto antes de correrlo en producción.

---
## 📄 Archivos raíz del proyecto

### `requirements.txt`
Lista todas las **librerías de Python** que necesita el proyecto para funcionar.
```
requests
beautifulsoup4
playwright
pandas
duckdb
spacy
mcp
streamlit
plotly
python-dotenv
pyyaml
```
Cualquier persona puede instalar todo con `pip install -r requirements.txt`

---

### `.env`
Archivo que guarda las **variables de entorno sensibles**: API keys, credenciales, URLs de bases de datos. **Nunca se sube a GitHub**.

---

### `.gitignore`
Le dice a Git qué archivos **ignorar y no subir al repositorio**: el `.env`, la carpeta `data/raw/`, archivos de caché de Python (`__pycache__`), etc.

---

### `README.md`
La **carta de presentación del proyecto**. Explica qué hace el proyecto, cómo instalarlo y correrlo, qué tecnologías usa y cómo está estructurado. Es lo primero que lee alguien que llega al repositorio.

---
## 🔄 Flujo completo del pipeline

```
1. run_pipeline.py arranca el proceso
       ↓
2. scraper_linkedin.py + scraper_indeed.py
   extraen ofertas → guardan en data/raw/
       ↓
3. clean_jobs.py limpia los datos
   extract_skills.py extrae las habilidades
   → guardan en data/processed/
       ↓
4. db.py inserta los datos en DuckDB
       ↓
5a. mcp_server/server.py expone los datos a Claude
    → Claude puede responder preguntas en lenguaje natural

5b. dashboard/app.py visualiza las tendencias
    → El usuario ve gráficas interactivas
```

---
## 🛠️ Stack tecnológico

| Herramienta | Uso en el proyecto |
|-------------|--------------------|
| Python | Lenguaje principal |
| BeautifulSoup / Playwright | Scraping de ofertas de trabajo |
| Pandas | Manipulación y limpieza de datos |
| DuckDB | Base de datos analítica local |
| spaCy / regex | Extracción de skills del texto |
| MCP (Anthropic) | Servidor para conectar Claude a los datos |
| Streamlit | Dashboard interactivo |
| Plotly | Gráficas del dashboard |
| APScheduler | Programar el pipeline para correr diario |

---
## ✅ Resumen

Cada carpeta tiene una responsabilidad clara y separada:

- **data/** → donde viven los datos (raw → processed)
- **notebooks/** → exploración y análisis
- **src/** → todo el código del pipeline
- **mcp_server/** → la capa de IA con MCP
- **dashboard/** → la visualización
- **config/** → la configuración centralizada
- **tests/** → las pruebas automáticas

Esta separación se llama **separación de responsabilidades** y es una buena práctica de ingeniería de software que hace el proyecto mantenible, escalable y fácil de entender.